In [ ]:
import torch
print(torch.__version__)
!nvcc --version
!python --version

In [ ]:
!git clone -b regularize-reward https://github.com/okcze/GFlowNet-CombOpt.git

In [ ]:
%%capture
!pip install hydra-core omegaconf submitit hydra-submitit-launcher einops ipdb
if torch.cuda.is_available():
    # !pip install  dgl -f https://data.dgl.ai/wheels/torch-2.4/cu121/repo.html
    !pip install --pre dgl -f https://data.dgl.ai/wheels-test/torch-2.4/cu121/repo.html
    # !pip install  --force-reinstall dglgo -f https://data.dgl.ai/wheels-test/repo.html
else:
    !pip install dgl==1.1.3
!pip install --force-reinstall -v "networkx==2.8.8"
# !pip install --upgrade hydra-core

In [ ]:
# Imports after restart
import os
import torch
import numpy as np
import datetime
import shutil
import zipfile
from google.colab import drive
import itertools
import dgl
import hydra

In [ ]:
print(torch.__version__)
print(dgl.__version__)

In [ ]:
drive.mount('/content/drive')

In [ ]:
%cd /content/GFlowNet-CombOpt/

In [ ]:
with zipfile.ZipFile('/content/drive/MyDrive/master_data/test_500_mis.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/test_500_mis')
with zipfile.ZipFile('/content/drive/MyDrive/master_data/train_4000_mis.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/train_4000_mis')

In [ ]:
os.makedirs('/content/GFlowNet-CombOpt/data/rb200-300/', exist_ok=True)
! cp -r /content/test_500_mis/content/GFlowNet-CombOpt/data/rb200-300/test /content/GFlowNet-CombOpt/data/rb200-300/test
! cp -r /content/train_4000_mis/content/GFlowNet-CombOpt/data/rb200-300/train /content/GFlowNet-CombOpt/data/rb200-300/train

In [ ]:
%%capture
# ! python ./data/rbgraph_generator.py --num_graph 500 --graph_type small --save_dir ./data/rb200-300/train
# ! python ./data/rbgraph_generator.py --num_graph 500 --graph_type small --save_dir ./data/rb200-300/test

In [ ]:
### SET CONFIG
task = "mis"
ref_alg = "mis_greedy"
NUM_EPOCHS = 10
current_date = "2025-03-30"

In [ ]:
# Parameters
frac_values = [0.25, 0.5]
boost_values = [5, 10, 50, 100, 1000]

# Generate grid search configurations
grid_search_params = [{'frac': frac, 'boost': boost} for frac, boost in itertools.product(frac_values, boost_values)]
grid_search_params += [{'frac': 0, 'boost': 1}]

# Get the current date for result storage
current_date = datetime.datetime.now().strftime("%Y-%m-%d")

# Iterate through all grid search parameters
for params in grid_search_params:
    frac_replaced = params['frac']
    boost = params['boost']
    run_name = f"frac_{frac_replaced:.2f}".replace(".", "_") + f"_boost_{boost}".replace(".", "_")

    # Run the command
    if torch.cuda.is_available():
        ! python -m gflownet.regularized_main task={task} ref_alg={ref_alg} frac_replaced={frac_replaced} reward_boost={boost} run_name={run_name} epochs={NUM_EPOCHS}
    else:
        ! python -m gflownet.regularized_main d=-1 task={task} ref_alg={ref_alg} frac_replaced={frac_replaced} reward_boost={boost} run_name={run_name} epochs={NUM_EPOCHS}

    # Create directory on Google Drive
    drive_path = f'/content/drive/MyDrive/master_data/{current_date}/{task}'
    if not os.path.exists(drive_path):
        os.makedirs(drive_path)

    ! cp -r /content/results/{run_name}/ {drive_path}/{run_name}/
    ! cp -r /content/GFlowNet-CombOpt/outputs/{run_name}/alg_best.pt {drive_path}/{run_name}/alg_best.pt
    ! cp -r /content/GFlowNet-CombOpt/outputs/{run_name}/alg.pt {drive_path}/{run_name}/alg.pt
